# GSM Smartphone Price Prediction and Value Recommendation Modeling

This notebook reproduces the modeling part of the DS Team 7 term project by using the preprocessed GSM CSV files stored in `content/`.

Main goals:
- Train regression models to predict `target_price_eur` and explain key price drivers.
- Analyze brand importance and adjusted brand premium under similar hardware conditions.
- Use clustering to divide phones into Entry, Mid-range, and Flagship market segments.
- Detect value-for-money outliers using `performance_to_price_ratio`, `expected_market_price_eur`, and `market_price_discount_ratio`.
- Recommend the best devices under a user budget by maximizing Performance-to-Price Ratio.

The comments in each code cell focus on the modeling step, the modeling reason, and the expected output.


In [ ]:
# 1. Import libraries and automatically locate the GitHub project root
# Why this cell exists:
# - The notebook should run both locally and in Google Colab.
# - The modeling code must use the repository folder structure directly, not a ZIP file or manual file picker.
# - Finding the root automatically reduces path errors when teammates or graders run the notebook.
# Expected structure: DS_Team7/content, DS_Team7/Preprocessing, DS_Team7/inspection_data, DS_Team7/modeling.
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Use a stable plotting configuration so generated figures look consistent in Colab and local environments.
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

# The repository URL can be overridden with an environment variable if the team uses a fork.
REPO_URL = os.environ.get("DS_TEAM7_REPO_URL", "https://github.com/zwonhong/DS_Team7.git")

# These marker files prove that the current folder is the final DS_Team7 project root.
# We check both modeling scripts and preprocessed CSV files because both are required to reproduce results.
PROJECT_MARKERS = [
    Path("content/gsm_processed_all(price_prediction).csv"),
    Path("content/gsm_processed(price_prediction).csv"),
    Path("content/gsm_processed_all(recommendation).csv"),
    Path("content/gsm_processed(recommendation).csv"),
    Path("modeling/run_modeling.py"),
]


def has_project_markers(path: Path) -> bool:
    """Return True only when all required modeling inputs exist under a candidate root."""
    return all((path / marker).exists() for marker in PROJECT_MARKERS)


def safe_resolve(path: Path) -> Path | None:
    """Resolve a path safely because some candidate paths may not exist in Colab."""
    try:
        return path.expanduser().resolve()
    except Exception:
        return None


def candidate_roots() -> list[Path]:
    """Return likely DS_Team7 repository roots without using ZIP files.

    Why this function searches several places:
    - In Colab, the repository is often under /content/DS_Team7.
    - Locally, the notebook may be opened from the project root, a subfolder, or a parent homework folder.
    - We include DS_git/DS_Team7 because the team reorganized paths into that structure during the project.
    """
    cwd = Path.cwd().resolve()
    env_root = os.environ.get("DS_TEAM7_ROOT")
    bases: list[Path] = []
    if env_root:
        bases.append(Path(env_root))
    bases.extend([
        cwd,
        cwd / "DS_Team7",
        cwd / "DS_Team7-main",
        cwd / "DS_git" / "DS_Team7",
        Path("/content/DS_Team7"),
        Path("/content/DS_Team7-main"),
    ])
    for parent in cwd.parents:
        if str(parent) in {"/", "/Users", "/home", "/private", "/var"}:
            break
        bases.extend([
            parent,
            parent / "DS_Team7",
            parent / "DS_Team7-main",
            parent / "DS_git" / "DS_Team7",
        ])
    seen = set()
    unique = []
    for base in bases:
        resolved = safe_resolve(base)
        if resolved is not None and resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def find_project_root() -> Path | None:
    """Return the first candidate folder that contains all required project markers."""
    for candidate in candidate_roots():
        if has_project_markers(candidate):
            return candidate
    return None


def clone_repo_for_colab() -> Path | None:
    """Clone the GitHub repository in Colab when only the notebook was opened.

    Why cloning is allowed here:
    - The notebook should be runnable by a grader without uploading every file manually.
    - The project still uses the GitHub folder structure directly after cloning; it does not load data from ZIP files.
    """
    content_root = Path("/content")
    if not content_root.exists():
        return None
    target = content_root / "DS_Team7"
    if not target.exists():
        print(f"GitHub repository clone: {REPO_URL} -> {target}")
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(target)],
            text=True,
            capture_output=True,
        )
        if result.returncode != 0:
            print(result.stdout)
            print(result.stderr)
            return None
    else:
        print(f"Using existing Colab repository candidate: {target}")
    return target if has_project_markers(target) else None


ROOT_DIR = find_project_root()
if ROOT_DIR is None:
    ROOT_DIR = clone_repo_for_colab()

if ROOT_DIR is None:
    searched = "\n".join(f"- {p}" for p in candidate_roots())
    raise FileNotFoundError(
        "Could not find the DS_Team7 GitHub-style project root.\n\n"
        "Required structure:\n"
        "DS_Team7/content/*.csv\n"
        "DS_Team7/Preprocessing/\n"
        "DS_Team7/inspection_data/\n"
        "DS_Team7/modeling/run_modeling.py\n\n"
        "This notebook is configured not to use ZIP files. "
        "If you opened only the notebook in Colab, this cell should clone the GitHub repository first. "
        "If cloning still fails, check that the main branch contains both modeling/ and content/ files.\n\n"
        f"Searched locations:\n{searched}"
    )

MODELING_DIR = ROOT_DIR / "modeling"
OUTPUT_DIR = MODELING_DIR / "outputs"
PLOT_DIR = OUTPUT_DIR / "plots"
CONTENT_DIR = ROOT_DIR / "content"
PREPROCESSING_DIR = ROOT_DIR / "Preprocessing"
INSPECTION_DIR = ROOT_DIR / "inspection_data"

# Move to the project root so every later relative path matches the GitHub layout.
os.chdir(ROOT_DIR)

print("ROOT_DIR:", ROOT_DIR)
print("CONTENT_DIR:", CONTENT_DIR, CONTENT_DIR.exists())
print("MODELING_DIR:", MODELING_DIR, MODELING_DIR.exists())
print("PREPROCESSING_DIR:", PREPROCESSING_DIR, PREPROCESSING_DIR.exists())
print("INSPECTION_DIR:", INSPECTION_DIR, INSPECTION_DIR.exists())
print("required content files exist:", all((CONTENT_DIR / name).exists() for name in [
    "gsm_processed_all(price_prediction).csv",
    "gsm_processed(price_prediction).csv",
    "gsm_processed_all(recommendation).csv",
    "gsm_processed(recommendation).csv",
]))


In [ ]:
# 2. Verify that the preprocessed modeling datasets are available
# Why this check is important:
# - The modeling part must use the preprocessing team's final CSV files.
# - Shape/null/object-column checks quickly show whether the final encoded files are model-ready.
# - If a required CSV is missing, it is better to fail early before running a long model pipeline.
required_members = [
    "gsm_processed_all(price_prediction).csv",
    "gsm_processed(price_prediction).csv",
    "gsm_processed_all(recommendation).csv",
    "gsm_processed(recommendation).csv",
]

if not CONTENT_DIR.exists():
    raise FileNotFoundError(f"content folder does not exist: {CONTENT_DIR}")

missing = [member for member in required_members if not (CONTENT_DIR / member).exists()]
if missing:
    raise FileNotFoundError(
        "Required preprocessed CSV files are missing: " + ", ".join(missing) +
        "\nThe DS_Team7/content/ folder must contain all required CSV files. In Colab, check whether Cell 1 cloned the GitHub repository correctly."
    )

summary_rows = []
print("content files:", sorted(p.name for p in CONTENT_DIR.glob("*.csv")))
for member in required_members:
    df = pd.read_csv(CONTENT_DIR / member)
    summary_rows.append({
        "file": f"content/{member}",
        "rows": df.shape[0],
        "cols": df.shape[1],
        "null_total": int(df.isna().sum().sum()),
        "object_cols": len(df.select_dtypes(include=["object", "string"]).columns),
    })

# The encoded/scaled "all" files should have zero object columns because machine learning models need numeric input.
display(pd.DataFrame(summary_rows))


## Modeling Libraries and Methods

- `lightgbm.LGBMRegressor`: a gradient boosting tree regressor. We used parameters such as `n_estimators`, `learning_rate`, `num_leaves`, `subsample`, and `colsample_bytree` to control tree count, learning speed, model complexity, and sampling. It was used because boosted trees usually perform well on structured tabular data with many encoded smartphone features.
- `RandomForestRegressor`: an ensemble of decision trees trained through bagging. It was used because smartphone price can depend on nonlinear interactions among RAM, storage, camera, display, battery, and brand.
- `KMeans`: a clustering algorithm that assigns observations to k centroids. We used `n_clusters=3`, `n_init=10`, and `random_state=42` to create Entry, Mid-range, and Flagship segments.
- `AgglomerativeClustering`: a hierarchical clustering method. We used `linkage='ward'` because it minimizes within-cluster variance and helps compare the KMeans segmentation.
- `IsolationForest`: an anomaly detection model. We used `contamination=0.05` to find unusual value-for-money candidates, then filtered them with high value score so the anomalies represent good deals rather than simply unusual devices.
- `PCA`: dimensionality reduction used only for visualization of clusters. The actual clustering is still performed on the full scaled feature set.
- `joblib`: used to save and reload trained model artifacts so the two-way demo can run without retraining.


In [ ]:
# 3. Run the full modeling pipeline
# Why this notebook calls run_modeling.py instead of duplicating all code:
# - The project keeps the actual implementation in one reproducible Python script.
# - The notebook acts as a Colab-friendly runner and result viewer.
# - This prevents the notebook and .py file from drifting into different versions.
# Pipeline order: Regression -> Feature Importance -> Brand Premium -> Clustering -> Value Outlier Detection -> Recommendation.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(MODELING_DIR / "run_modeling.py"), "--run"],
    cwd=str(ROOT_DIR),
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("run_modeling.py failed")


In [ ]:
# 4. Review Task A: regression model performance
# Why these metrics are shown:
# - MAE and RMSE measure price error in EUR, which is easy to interpret.
# - sMAPE gives a relative percentage-style error across cheap and expensive phones.
# - MASE compares the model against a simple median-price baseline.
# - R2 shows how much price variance the model explains.
# - Fold-level CV results prove that 5-fold cross-validation was actually executed.
holdout = pd.read_csv(OUTPUT_DIR / "regression_holdout_metrics.csv")
cv = pd.read_csv(OUTPUT_DIR / "regression_cv_metrics.csv")
cv_fold = pd.read_csv(OUTPUT_DIR / "regression_cv_fold_metrics.csv")
print("Holdout metrics")
display(holdout.head(8))
print("5-fold CV average metrics")
display(cv.head(8))
print("5-fold CV fold-level metrics sample")
display(cv_fold.head(10))


In [ ]:
# 5. Review price drivers and adjusted brand premium
# Why this matters:
# - Feature importance explains which hardware and brand variables affect price prediction.
# - Adjusted brand premium compares actual price with spec-only predicted price, helping isolate brand effects under similar specifications.
importance = pd.read_csv(OUTPUT_DIR / "feature_importance_best_target.csv")
brand_premium = pd.read_csv(OUTPUT_DIR / "adjusted_brand_premium_summary.csv")
print("Top feature importance")
display(importance.sort_values("importance", ascending=False).head(12))
print("Adjusted brand premium")
display(brand_premium.head(12))


In [ ]:
# 6. Review Task B: clustering-based market segments
# Why clustering is used:
# - It groups phones into market tiers without manually labeling every device.
# - Entry, Mid-range, and Flagship segments make later value-outlier detection fairer because phones are compared within similar market levels.
cluster_summary = pd.read_csv(OUTPUT_DIR / "cluster_segment_summary.csv")
cluster_quality = pd.read_csv(OUTPUT_DIR / "cluster_algorithm_comparison.csv")
display(cluster_summary)
display(cluster_quality)


In [ ]:
# 7. Review Task C: value-for-money outliers and budget recommendations
# Why these validation tables are shown:
# - Value outliers should be cheaper, have higher PPR, and be priced below expected market price.
# - Recommendation outputs must respect the user budget constraint.
# - The recommendation ranking should be sorted by Performance-to-Price Ratio in descending order.
outlier_validation = pd.read_csv(OUTPUT_DIR / "value_outlier_validation_summary.csv")
outlier_methods = pd.read_csv(OUTPUT_DIR / "value_outlier_method_comparison.csv")
rec_validation = pd.read_csv(OUTPUT_DIR / "recommendation_constraint_validation.csv")
recommendations = pd.read_csv(OUTPUT_DIR / "recommendations_all_scenarios.csv")
display(outlier_methods)
display(outlier_validation)
display(rec_validation)
display(recommendations.head(20))


In [ ]:
# 8. Check key visualization files
# Why plots are checked here:
# - Plots make it easier to inspect model behavior beyond numeric tables.
# - These figures show feature importance, clustering, outliers, and recommendation lift.
plot_files = sorted(PLOT_DIR.glob("*.png"))
for path in plot_files:
    print(path.name)
if plot_files:
    display(Image(filename=str(PLOT_DIR / "lgbm_feature_importance_top15.png")))


In [ ]:
# 9. Test the executable two-way solution
# Why this cell is included:
# - It verifies that the saved model artifacts can be reused without retraining.
# - The demo confirms both stakeholder outputs: a company-side price guide and a customer-side recommendation list.
result = subprocess.run(
    [sys.executable, str(MODELING_DIR / "two_way_solution.py"), "--mode", "demo"],
    cwd=str(MODELING_DIR),
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("two_way_solution.py demo failed")


## Conclusion

- Company side: the regression model and feature-importance outputs provide a fair-price guideline and explain which hardware/brand factors drive smartphone price.
- Customer side: clustering segments, expected-market-price underpricing, value outlier detection, and budget-constrained PPR ranking recommend practical alternatives under a user budget.
- Modeling workflow: the notebook confirms scaling/encoding, regression, clustering, value-outlier detection, budget recommendation, and reusable output files.

Limitations: prices are EUR-based, hardware specs come from text parsing, regional/channel price differences are not modeled, and old/new phones coexist in the dataset. Because the final GSM data does not provide a direct CPU benchmark, this project uses RAM, storage, camera, battery, network, display features, and `spec_score_0_100` as proxy hardware-performance indicators.
